# OverView
CV向上を目指してカラムを消す。

In [15]:
# 基本
import numpy as np
import pandas as pd

# 可視化
import matplotlib.pyplot as plt
import seaborn as sns

# 機械学習
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import root_mean_squared_error

# 表示設定
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# モデル
import lightgbm as lgb
import catboost as cb

train_df = pd.read_csv('../data/train.csv')
test_df = pd.read_csv('../data/test.csv')

train_df.shape, test_df.shape

((1460, 81), (1459, 80))

# データ加工

In [16]:
# 外れ値の削除
train_df = train_df[~train_df['Id'].isin([1299, 524])].copy()

In [17]:
n_train = len(train_df)
data = pd.concat([train_df, test_df])

In [18]:
#dealing with missing data
drop_columns = [
    'PoolQC',
    'MiscFeature',
    'Alley',
    'Fence',
    'MasVnrType',
    'FireplaceQu',
    'LotFrontage',
    'GarageQual',
    'GarageFinish',
    'GarageType',
    'GarageYrBlt',
    'GarageCond',
    'BsmtFinType2',
    'BsmtExposure',
    'BsmtCond',
    'BsmtQual',
    'BsmtFinType1',
    'MasVnrArea'
]
data = data.drop(columns=drop_columns)

In [19]:
# カテゴリ列をcategory型にする
cat_cols = data.select_dtypes(include=["object", "str"]).columns
for col in cat_cols:
  data[col] = data[col].astype("category")

In [20]:
# 家全体の広さを特徴量にする
data['TotalSF'] = (
  data['TotalBsmtSF']
  + data['1stFlrSF']
  + data['2ndFlrSF']
)

In [21]:
train_df = data.iloc[:n_train].copy()
test_df = data.iloc[n_train:].copy()
test_df = test_df.drop(columns=["SalePrice"])

train_df.shape, test_df.shape

((1458, 64), (1459, 63))

In [22]:
X_train = train_df.drop(columns='SalePrice')
y_train = np.log1p(train_df['SalePrice'])

X_train.shape, y_train.shape

((1458, 63), (1458,))

In [23]:
# catboost用にカテゴリ列の欠損を"Missing"で埋めたデータを用意する
cat_cols = X_train.select_dtypes(include=["category"]).columns.tolist()

train_cat_df = train_df.copy()
test_cat_df = test_df.copy()

data = pd.concat([train_cat_df, test_cat_df])

for col in cat_cols:
  data[col] = data[col].cat.add_categories(['Missing']).fillna("Missing")

train_cat_df = data.iloc[:len(train_cat_df)].copy()
test_cat_df = data.iloc[len(train_cat_df):].copy()
test_cat_df = test_cat_df.drop(columns="SalePrice")

train_cat_df.shape, test_cat_df.shape

((1458, 64), (1459, 63))

In [24]:
X_train_cat = train_cat_df.drop(columns="SalePrice")
y_train_cat = np.log1p(train_cat_df['SalePrice'])

X_train_cat.shape, y_train_cat.shape

((1458, 63), (1458,))

# Training

In [25]:
# LightGBM
params = {
  "boosting_type": "gbdt",
  "objective": "regression",
  "metric": "rmse",
  "num_iterations": 1000,
  "learning_rate": 0.02,
  "num_leaves": 16,
  "max_depth": -1,
  "min_data_in_leaf": 20,
  "min_sum_hessian_in_leaf": 1e-3,
  "bagging_fraction": 0.9,
  "bagging_freq": 1,
  "feature_fraction": 0.9,
  "lambda_l1": 0.0,
  "lambda_l2": 0.0,
  "random_state": 42,
  "verbosity": -1
}

lgb_model = lgb.LGBMRegressor(**params)
lgb_model.fit(X_train, y_train)

,num_leaves,16
,learning_rate,0.02
,objective,'regression'
,random_state,42
,metric,'rmse'
,num_iterations,1000
,min_data_in_leaf,20
,min_sum_hessian_in_leaf,0.001
,bagging_fraction,0.9
,bagging_freq,1
,feature_fraction,0.9


In [26]:
# Cat Boost
cat_model = cb.CatBoostRegressor(
  iterations=1000,
  learning_rate=0.03,
  depth=6,
  loss_function="RMSE",
  verbose=False,
  random_state=42
)

cat_model.fit(
  X_train_cat,
  y_train_cat,
  cat_features=cat_cols
)

CatBoostRegressor(depth=6, iterations=1000, learning_rate=0.03, loss_function='RMSE', random_state=42, verbose=False)

# CV

In [ ]:
# LightGMB
kf = KFold(
  n_splits=5,
  shuffle=True,
  random_state=42
)

scores = cross_val_score(
  lgb_model,
  X_train,
  y_train,
  cv=kf,
  scoring='neg_root_mean_squared_error'
)

rmse_scores = -scores

current_lgbm_rmse = rmse_scores.mean()
prev_lgbm_rmse = 0.120426927312

improvement = prev_lgbm_rmse - current_lgbm_rmse
improvement_rate = improvement / prev_lgbm_rmse * 100

print('-'*20)
print(f"Prev LGBM RMSE : {prev_lgbm_rmse:.12f}")
print(f"Current LGBM RMSE  : {current_lgbm_rmse:.12f}")
print(f"Improvement   : {improvement:+.12f}")
print(f"Improvement % : {improvement_rate:+.2f}%")
print('-'*20)

--------------------
Prev LGBM RMSE : 0.121397734512
Current LGBM RMSE  : 0.121397734512
Improvement   : -0.000000000000
Improvement % : -0.00%
--------------------


In [ ]:
# CatBoost
kf = KFold(
  n_splits=5,
  shuffle=True,
  random_state=42
)

scores = cross_val_score(
  cat_model,
  X_train_cat,
  y_train_cat,
  cv=kf,
  scoring='neg_root_mean_squared_error',
  params={
    "cat_features": cat_cols  # cross_val_score内でfitするときにカテゴリ列のリストが必要
  }
)

rmse_scores = -scores
print(rmse_scores)

# base lineとの比較
current_cat_rmse = rmse_scores.mean()
prev_cat_rmse = 0.115873836247

improvement = prev_cat_rmse - current_cat_rmse
improvement_rate = improvement / prev_cat_rmse * 100

print('-'*20)
print(f"Prev CAT RMSE : {prev_cat_rmse:.12f}")
print(f"Current CAT RMSE  : {current_cat_rmse:.12f}")
print(f"Improvement   : {improvement:+.12f}")
print(f"Improvement % : {improvement_rate:+.2f}%")
print('-'*20)

KeyboardInterrupt: 

# Ensemble

In [ ]:
LGBM_WEIGHT = 0.5
CAT_BOOST_WEIGHT = 0.5

In [ ]:
kf = KFold(
  n_splits=5,
  shuffle=True,
  random_state=42
)

ensemble_scores = []

for train_idx, val_idx in kf.split(X_train):
  X_tr = X_train.iloc[train_idx]
  X_va   = X_train.iloc[val_idx]

  y_tr = y_train.iloc[train_idx]
  y_va   = y_train.iloc[val_idx]

  X_tr_cat = X_train_cat.iloc[train_idx]
  X_va_cat   = X_train_cat.iloc[val_idx]

  # Learning
  lgb_model.fit(X_tr, y_tr)

  cat_model.fit(
      X_tr_cat,
      y_tr,
      cat_features=cat_cols
  )

  # Prediction
  lgb_pred_log = lgb_model.predict(X_va)
  cat_pred_log = cat_model.predict(X_va_cat)

  # Ensemble
  ensemble_pred_log = (
    LGBM_WEIGHT * lgb_pred_log
    + CAT_BOOST_WEIGHT * cat_pred_log
  )

  # Scoring
  ensemble_rmse = root_mean_squared_error(
    y_va,
    ensemble_pred_log
  )

  ensemble_scores.append(ensemble_rmse)

# export logs
prev_ens_rmse = 0.114591533113
print(ensemble_scores)

current_ens_rmse = np.mean(ensemble_scores)

improvement = prev_ens_rmse - current_ens_rmse
improvement_rate = improvement / prev_ens_rmse * 100

print("===== ENSEMBLE CV RESULT =====")
print(f"Prev RMSE        : {prev_ens_rmse:.12f}")
print(f"Current RMSE     : {current_ens_rmse:.12f}")
print(f"Improvement      : {improvement:+.12f}")
print(f"Improvement rate : {improvement_rate:+.2f}%")

[0.1155886490870945, 0.10799709919485975, 0.11775889156584277, 0.12527633850944886, 0.10633668720770148]
===== ENSEMBLE CV RESULT =====
Prev RMSE        : 0.122330462986
Current RMSE     : 0.114591533113
Improvement      : +0.007738929873
Improvement rate : +6.33%


In [ ]:
lgb_model.fit(X_train, y_train)
cat_model.fit(
  X_train_cat,
  y_train_cat,
  cat_features=cat_cols
)

pred_lgb_log = lgb_model.predict(test_df)
pred_cat_log = cat_model.predict(test_cat_df)

pred = 0.5 * pred_lgb_log + 0.5 * pred_cat_log
pred = np.expm1(pred)

# Submission

In [ ]:
submission = pd.DataFrame({
    "Id": test_df["Id"],
    "SalePrice": pred
})

submission.to_csv("../submissions/submission.csv", index=False)

In [ ]:
submission.shape

(1459, 2)